# Hospital PPE Detector — Training Notebook

Trains a YOLOv8 model on the CPPE-5 dataset (medical PPE: Coverall,
Face_Shield, Gloves, Goggles, Mask) for the Hospital PPE Detector app.
Exports `best.pt` for local inference in VS Code.

Dataset source: https://huggingface.co/datasets/rishitdagli/cppe-5

In [ ]:
!pip install ultralytics datasets -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.1/46.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 79.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 6.9 MB/s eta 0:00:00


## Step 1: Load the CPPE-5 dataset from Hugging Face

CPPE-5 has 1000 training images and 29 test images, with COCO-format
bounding box annotations.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("rishitdagli/cppe-5")
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['image_id', 'image', 'width', 'height', 'objects'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['image_id', 'image', 'width', 'height', 'objects'],
        num_rows: 29
    })
})


## Step 2: Verify class labels

Confirming the true class-index mapping directly from the dataset's
own metadata, rather than assuming.

In [ ]:
print(dataset["train"].features)

{'image_id': Value('int64'), 'image': Image(mode=None, decode=True), 'width': Value('int32'), 'height': Value('int32'), 'objects': {'id': List(Value('int64')), 'area': List(Value('int64')), 'bbox': List(List(Value('float32'), length=4)), 'category': List(ClassLabel(names=['Coverall', 'Face_Shield', 'Gloves', 'Goggles', 'Mask']))}}


Confirmed mapping: 0 = Coverall, 1 = Face_Shield, 2 = Gloves, 3 = Goggles, 4 = Mask

## Step 3: Define the COCO → YOLO conversion function

CPPE-5 stores boxes as COCO format: `[x_min, y_min, width, height]` in
pixel coordinates. YOLO needs normalized `[class_id, x_center, y_center,
width, height]` values between 0 and 1.

In [ ]:
def convert_to_yolo_format(sample):
    img_width = sample["width"]
    img_height = sample["height"]
    boxes = sample["objects"]["bbox"]
    categories = sample["objects"]["category"]

    yolo_lines = []
    for box, cat_id in zip(boxes, categories):
        x_min, y_min, w, h = box
        x_center = (x_min + w / 2) / img_width
        y_center = (y_min + h / 2) / img_height
        norm_w = w / img_width
        norm_h = h / img_height
        yolo_lines.append(f"{cat_id} {x_center:.6f} {y_center:.6f} {norm_w:.6f} {norm_h:.6f}")

    return yolo_lines

Quick test — verify the conversion on one sample before processing the
full dataset.

In [ ]:
sample = dataset["train"][0]
lines = convert_to_yolo_format(sample)
for line in lines:
    print(line)

4 0.358961 0.203620 0.077413 0.078431
4 0.889183 0.171946 0.060445 0.042232
0 0.301166 0.511312 0.262990 0.929110
0 0.892895 0.404977 0.214210 0.604827


## Step 4: Convert and save the full dataset in YOLO format

Creates the folder structure YOLOv8 expects (`images/train`,
`labels/train`, `images/val`, `labels/val`), then saves every image
and its corresponding label file. Images are converted to RGB before
saving, since some source images are RGBA (which JPEG doesn't support).

In [ ]:
import os

for split in ["train", "val"]:
    os.makedirs(f"dataset/images/{split}", exist_ok = True)
    os.makedirs(f"dataset/labels/{split}", exist_ok = True)

def process_split(hf_split_name, output_split_name):
    split_data = dataset[hf_split_name]
    for idx, sample in enumerate(split_data):
        img = sample["image"]
        if img.mode != "RGB":
            img = img.convert("RGB")
        img_filename = f"{idx:04d}.jpg"
        img.save(f"dataset/images/{output_split_name}/{img_filename}")

        yolo_lines = convert_to_yolo_format(sample)
        label_filename = f"{idx:04d}.txt"
        with open(f"dataset/labels/{output_split_name}/{label_filename}", "w") as f:
            f.write("\n".join(yolo_lines))

process_split("train", "train")
process_split("test", "val")

print("Done. Train images:", len(os.listdir("dataset/images/train")))
print("Val images:", len(os.listdir("dataset/images/val")))

Done. Train images: 1000
Val images: 29


## Step 5: Create the data.yaml config file

This tells YOLOv8 where the images/labels live and what the class
names are, in the order matching their class IDs (0-4).

In [ ]:
yaml_content = """
train: /content/dataset/images/train
val: /content/dataset/images/val

nc: 5
names: ['Coverall', 'Face_Shield', 'Gloves', 'Goggles', 'Mask']
"""

with open("dataset/data.yaml", "w") as f:
    f.write(yaml_content)

print("data.yaml created:")
print(yaml_content)

data.yaml created:

train: /content/dataset/images/train
val: /content/dataset/images/val

nc: 5
names: ['Coverall', 'Face_Shield', 'Gloves', 'Goggles', 'Mask']



## Step 6: Train the YOLOv8 model

Fine-tuning a pretrained yolov8n (nano) model on our converted CPPE-5
dataset. Using nano for faster training given the small dataset size
(1000 images); a bigger base model isn't likely to help much here
and would train slower.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data = "dataset/data.yaml",
    epochs = 50,
    imgsz = 640,
    batch = 16,
    project = "ppe_training",
    name = "ppe_run1"
)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
Ultralytics 8.4.139 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscr

## Step 7: Save the trained model to Google Drive (permanent storage)

Colab's local storage is temporary and gets wiped when the session
ends. Saving best.pt to Drive immediately after training protects
against losing the trained weights.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
import os

os.makedirs("/content/drive/MyDrive/YOLO_CV_Project_Models", exist_ok=True)
shutil.copy("/content/runs/detect/ppe_training/ppe_run1/weights/best.pt",
            "/content/drive/MyDrive/YOLO_CV_Project_Models/ppe_best.pt")

print("Saved to Google Drive successfully.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved to Google Drive successfully.


In [ ]:
shutil.copy("/content/runs/detect/ppe_training/ppe_run1/results.png",
            "/content/drive/MyDrive/YOLO_CV_Project_Models/ppe_results.png")
print("Results chart saved too.")

Results chart saved too.
